# EMA Ribbon Regime Strategy — Ather Energy

Hourly Ather Energy candlesticks with two complementary ribbons:

- **Trend ribbon:** 144-period EMAs of high, close, and low.
- **Momentum ribbon:** 5-, 8-, 13-, and 21-period EMAs of close.

A bullish regime requires EMA 21 and the candle body to be above the trend ribbon; a bearish regime requires them to be below it.

`PRIMARY_SYMBOL` below drives this detailed single-symbol walkthrough (data, chart, backtest). A later section reruns the identical strategy across all five symbols in `data/raw/` as a basket.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from backtesting import Backtest, Strategy


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether Jupyter starts there or in a notebook folder."""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate the visualizer project root.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from visualizer import load_candle_json

pd.options.display.float_format = "{:,.2f}".format

/home/amit/anaconda3/envs/visualizer/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## Load stock data

In [2]:
SYMBOLS = {
    "adanigreen": {"display": "Adani Green Energy", "file": "adanigreen_2023-12-01_2026-08-19.json"},
    "atherenerg": {"display": "Ather Energy", "file": "atherenerg_2023-12-01_2026-08-19.json"},
    "hdfcbank": {"display": "HDFC Bank", "file": "hdfcbank_2023-12-01_2026-08-19.json"},
    "netweb": {"display": "Netweb Technologies", "file": "netweb_2023-12-01_2026-08-19.json"},
    "reliance": {"display": "Reliance Industries", "file": "reliance_2023-12-01_2026-08-19.json"},
}
PRIMARY_SYMBOL = "atherenerg"


def load_symbol_data(symbol_key: str) -> pd.DataFrame:
    """Load and clean one symbol's raw candle JSON into a timestamp-indexed frame."""
    data_path = PROJECT_ROOT / "data" / "raw" / SYMBOLS[symbol_key]["file"]
    df = load_candle_json(data_path)

    required_columns = {"timestamp", "open", "high", "low", "close", "volume"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")

    numeric_columns = ["open", "high", "low", "close", "volume"]
    df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
    df = (
        df.dropna(subset=["timestamp", "open", "high", "low", "close"])
        .sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .set_index("timestamp")
    )
    return df


price = load_symbol_data(PRIMARY_SYMBOL)

print(f"Loaded {SYMBOLS[PRIMARY_SYMBOL]['file']} ({SYMBOLS[PRIMARY_SYMBOL]['display']})")
print(f"{len(price):,} hourly candles from {price.index.min().date()} to {price.index.max().date()}")
display(price.head())

Loaded atherenerg_2023-12-01_2026-08-19.json (Ather Energy)
2,248 hourly candles from 2025-05-06 to 2026-08-19


,open,high,low,close,volume,open_interest
timestamp,,,,,,
2025-05-06 09:15:00+05:30,327.50,333.00,321.00,323.70,5493059,0
2025-05-06 10:15:00+05:30,323.70,325.75,308.39,312.80,3668403,0
2025-05-06 11:15:00+05:30,312.80,313.85,309.25,310.30,1118619,0
2025-05-06 12:15:00+05:30,310.20,311.70,306.64,309.75,852891,0
2025-05-06 13:15:00+05:30,309.75,310.00,307.50,309.20,581795,0


## Calculate trend and momentum ribbons

In [3]:
TREND_PERIOD = 144
MOMENTUM_PERIODS = (5, 8, 13, 21)
ADX_PERIOD = 8
ADX_THRESHOLD = 30
ATR_PERIOD = 10
CHANDELIER_LOOKBACK = 21          # Bars used to find the trailing highest-high / lowest-low
CHANDELIER_ATR_MULTIPLE = 3.0     # Trail distance behind that extreme, in ATRs


def add_ema_ribbon_indicators(
    df: pd.DataFrame,
    trend_period: int = TREND_PERIOD,
    momentum_periods: tuple[int, ...] = MOMENTUM_PERIODS,
    adx_period: int = ADX_PERIOD,
    adx_threshold: float = ADX_THRESHOLD,
    atr_period: int = ATR_PERIOD,
    chandelier_lookback: int = CHANDELIER_LOOKBACK,
    chandelier_atr_multiple: float = CHANDELIER_ATR_MULTIPLE,
) -> pd.DataFrame:
    """Add trend/momentum EMA ribbons, ADX/ATR, regime flags, and chandelier trail levels."""
    df = df.copy()

    for source in ("high", "close", "low"):
        df[f"ema_{trend_period}_{source}"] = df[source].ewm(
            span=trend_period,
            adjust=False,
            min_periods=trend_period,
        ).mean()

    for period in momentum_periods:
        df[f"ema_{period}_close"] = df["close"].ewm(
            span=period,
            adjust=False,
            min_periods=period,
        ).mean()

    df[f"adx_{adx_period}"] = ta.trend.adx(
        df["high"],
        df["low"],
        df["close"],
        window=adx_period,
    )

    df[f"atr_{atr_period}"] = ta.volatility.AverageTrueRange(
        high=df["high"],
        low=df["low"],
        close=df["close"],
        window=atr_period,
    ).average_true_range()

    trend_high = df[f"ema_{trend_period}_high"]
    trend_close = df[f"ema_{trend_period}_close"]
    trend_low = df[f"ema_{trend_period}_low"]
    ema_5 = df["ema_5_close"]
    ema_8 = df["ema_8_close"]
    ema_13 = df["ema_13_close"]
    ema_21 = df["ema_21_close"]
    adx = df[f"adx_{adx_period}"]
    previous_adx = adx.shift(1)
    adx_rising = adx.gt(previous_adx)

    df["bullish_regime"] = (
        adx.gt(adx_threshold)
        & adx_rising
        & ema_5.gt(ema_8)
        & ema_8.gt(ema_13)
        & ema_13.gt(ema_21)
        & ema_21.gt(trend_high)
        & df["open"].gt(trend_high)
        & df["close"].gt(trend_high)
    )
    df["bearish_regime"] = (
        adx.gt(adx_threshold)
        & adx_rising
        & ema_5.lt(ema_8)
        & ema_8.lt(ema_13)
        & ema_13.lt(ema_21)
        & ema_21.lt(trend_low)
        & df["open"].lt(trend_low)
        & df["close"].lt(trend_low)
    )
    df["trend"] = np.select(
        [df["bullish_regime"], df["bearish_regime"]],
        ["Bullish", "Bearish"],
        default="Neutral",
    )

    df["bullish_crossover"] = df["bullish_regime"] & ~df["bullish_regime"].shift(1, fill_value=False)
    df["bearish_crossover"] = df["bearish_regime"] & ~df["bearish_regime"].shift(1, fill_value=False)
    df["crossover"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[df["bullish_crossover"], "crossover"] = "Bullish"
    df.loc[df["bearish_crossover"], "crossover"] = "Bearish"

    df["momentum"] = np.select(
        [
            ema_5.gt(ema_8) & ema_8.gt(ema_13) & ema_13.gt(ema_21),
            ema_5.lt(ema_8) & ema_8.lt(ema_13) & ema_13.lt(ema_21),
        ],
        ["Bullish", "Bearish"],
        default="Mixed",
    )

    atr = df[f"atr_{atr_period}"]
    df["chandelier_long"] = (
        df["high"].rolling(chandelier_lookback, min_periods=chandelier_lookback).max()
        - chandelier_atr_multiple * atr
    )
    df["chandelier_short"] = (
        df["low"].rolling(chandelier_lookback, min_periods=chandelier_lookback).min()
        + chandelier_atr_multiple * atr
    )

    return df


price = add_ema_ribbon_indicators(price)

latest = price.iloc[-1]
print(
    f"Latest close: ₹{latest['close']:,.2f} | "
    f"EMA 21: ₹{latest['ema_21_close']:,.2f} | "
    f"EMA {TREND_PERIOD} band: ₹{latest[f'ema_{TREND_PERIOD}_low']:,.2f}–"
    f"₹{latest[f'ema_{TREND_PERIOD}_high']:,.2f} | "
    f"Trend: {latest['trend']} | Momentum: {latest['momentum']} | "
    f"ADX {ADX_PERIOD}: {latest[f'adx_{ADX_PERIOD}']:.2f} | "
    f"ATR {ATR_PERIOD}: ₹{latest[f'atr_{ATR_PERIOD}']:,.2f}"
)

Latest close: ₹1,440.00 | EMA 21: ₹1,470.20 | EMA 144 band: ₹1,395.24–₹1,413.43 | Trend: Neutral | Momentum: Bearish | ADX 8: 40.32 | ATR 10: ₹13.23


## Ribbon entry events

An event is recorded only on the first bar entering a bullish or bearish regime, rather than on every bar that remains in that regime.

In [4]:
signal_columns = [
    "close",
    *(f"ema_{period}_close" for period in MOMENTUM_PERIODS),
    f"ema_{TREND_PERIOD}_low",
    f"ema_{TREND_PERIOD}_close",
    f"ema_{TREND_PERIOD}_high",
    "momentum",
    "crossover",
]
crossover_events = (
    price.loc[
        price["bullish_crossover"] | price["bearish_crossover"],
        signal_columns,
    ]
    .rename_axis("timestamp")
    .reset_index()
)

print(
    f"Found {price['bullish_crossover'].sum()} bullish and "
    f"{price['bearish_crossover'].sum()} bearish entry event(s). "
    f"Showing the latest {min(20, len(crossover_events))}."
)
display(crossover_events.tail(20))

Found 88 bullish and 11 bearish entry event(s). Showing the latest 20.


,timestamp,close,ema_5_close,ema_8_close,ema_13_close,ema_21_close,ema_144_low,ema_144_close,ema_144_high,momentum,crossover
79,2026-06-09 14:15:00+05:30,"1,035.55","1,023.10","1,020.94","1,019.04","1,014.86",950.61,958.05,966.53,Bullish,Bullish
80,2026-06-22 13:15:00+05:30,963.70,968.15,969.26,971.08,975.53,975.61,982.50,990.28,Bearish,Bearish
81,2026-06-29 10:15:00+05:30,"1,026.60","1,011.79","1,007.56","1,002.81",997.61,978.46,985.58,992.98,Bullish,Bullish
82,2026-07-02 14:15:00+05:30,"1,142.10","1,135.94","1,134.09","1,129.57","1,117.30","1,017.04","1,025.35","1,033.37",Bullish,Bullish
83,2026-07-06 11:15:00+05:30,"1,138.80","1,136.72","1,135.83","1,134.45","1,129.05","1,032.94","1,040.93","1,049.02",Bullish,Bullish
84,2026-07-07 12:15:00+05:30,"1,149.50","1,143.46","1,140.60","1,138.08","1,134.07","1,043.11","1,051.02","1,059.01",Bullish,Bullish
85,2026-07-08 09:15:00+05:30,"1,159.30","1,145.78","1,143.58","1,141.12","1,137.33","1,048.11","1,056.04","1,064.10",Bullish,Bullish
86,2026-07-08 15:15:00+05:30,"1,204.00","1,184.41","1,175.59","1,166.02","1,156.34","1,057.49","1,065.88","1,074.46",Bullish,Bullish
87,2026-07-13 11:15:00+05:30,"1,214.50","1,217.12","1,217.08","1,214.30","1,206.17","1,089.60","1,098.13","1,106.79",Bullish,Bullish
88,2026-07-15 09:15:00+05:30,"1,285.40","1,222.65","1,211.99","1,206.65","1,203.25","1,103.95","1,113.53","1,122.48",Bullish,Bullish


## EMA ribbon chart

The purple band is the EMA-144 high/low trend ribbon, with its close-based EMA as a dotted centerline. The green-to-red lines form the fast-to-slow momentum ribbon. Triangles mark regime entries. Below price are separate **Volume** and **ADX 8** panels; the dotted ADX line marks the trend-strength threshold at 30. Trading observations are evenly spaced so overnight, weekend, and holiday gaps are collapsed.

In [5]:
# Use None for the full history; a recent window keeps hourly candles readable.
PLOT_BARS = 500
symbol_display = SYMBOLS[PRIMARY_SYMBOL]["display"]
plot_df = price.copy() if PLOT_BARS is None else price.tail(PLOT_BARS).copy()
x_values = plot_df.index.strftime("%Y-%m-%d %H:%M")

bullish = plot_df.loc[plot_df["bullish_crossover"]]
bearish = plot_df.loc[plot_df["bearish_crossover"]]
marker_padding = max(float((plot_df["high"] - plot_df["low"]).median()) * 0.55, 1.0)
volume_colors = np.where(
    plot_df["close"].ge(plot_df["open"]),
    "rgba(22, 163, 74, 0.42)",
    "rgba(220, 38, 38, 0.42)",
)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.58, 0.16, 0.14, 0.12],
    vertical_spacing=0.035,
)

# Draw the trend ribbon first so candles and momentum lines stay crisp above it.
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=plot_df[f"ema_{TREND_PERIOD}_low"],
        mode="lines",
        name=f"EMA {TREND_PERIOD} Low",
        line=dict(color="rgba(109, 40, 217, 0.70)", width=1.2),
        hovertemplate=f"EMA {TREND_PERIOD} low: ₹%{{y:,.2f}}<extra></extra>",
        showlegend=False,
        legendgroup="trend",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=plot_df[f"ema_{TREND_PERIOD}_high"],
        mode="lines",
        name=f"EMA {TREND_PERIOD} High–Low",
        line=dict(color="rgba(109, 40, 217, 0.70)", width=1.2),
        fill="tonexty",
        fillcolor="rgba(124, 58, 237, 0.13)",
        hovertemplate=f"EMA {TREND_PERIOD} high: ₹%{{y:,.2f}}<extra></extra>",
        legendgroup="trend",
        legendrank=20,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=plot_df[f"ema_{TREND_PERIOD}_close"],
        mode="lines",
        name=f"EMA {TREND_PERIOD} Close",
        line=dict(color="#6d28d9", width=1.6, dash="dot"),
        hovertemplate=f"EMA {TREND_PERIOD} close: ₹%{{y:,.2f}}<extra></extra>",
        legendgroup="trend",
        legendrank=21,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Candlestick(
        x=x_values,
        open=plot_df["open"],
        high=plot_df["high"],
        low=plot_df["low"],
        close=plot_df["close"],
        name=symbol_display,
        increasing_line_color="#16a34a",
        decreasing_line_color="#dc2626",
        legendrank=10,
    ),
    row=1,
    col=1,
)

momentum_colors = {5: "#16a34a", 8: "#65a30d", 13: "#f59e0b", 21: "#dc2626"}
for legend_rank, period in enumerate(MOMENTUM_PERIODS, start=30):
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=plot_df[f"ema_{period}_close"],
            mode="lines",
            name=f"EMA {period}",
            line=dict(
                color=momentum_colors[period],
                width=2.0 if period in (5, 21) else 1.5,
            ),
            hovertemplate=f"EMA {period}: ₹%{{y:,.2f}}<extra></extra>",
            legendgroup="momentum",
            legendrank=legend_rank,
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=bullish.index.strftime("%Y-%m-%d %H:%M"),
        y=bullish["low"] - marker_padding,
        mode="markers",
        name="Bullish entry",
        customdata=bullish[["close", "ema_21_close", f"ema_{TREND_PERIOD}_high", "momentum"]],
        marker=dict(
            symbol="triangle-up",
            color="#15803d",
            size=13,
            line=dict(color="white", width=1),
        ),
        hovertemplate=(
            "%{x}<br><b>Bullish regime begins</b>"
            "<br>Close: ₹%{customdata[0]:,.2f}"
            "<br>EMA 21: ₹%{customdata[1]:,.2f}"
            f"<br>EMA {TREND_PERIOD} high: ₹%{{customdata[2]:,.2f}}"
            "<br>Momentum: %{customdata[3]}<extra></extra>"
        ),
        legendrank=40,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=bearish.index.strftime("%Y-%m-%d %H:%M"),
        y=bearish["high"] + marker_padding,
        mode="markers",
        name="Bearish entry",
        customdata=bearish[["close", "ema_21_close", f"ema_{TREND_PERIOD}_low", "momentum"]],
        marker=dict(
            symbol="triangle-down",
            color="#b91c1c",
            size=13,
            line=dict(color="white", width=1),
        ),
        hovertemplate=(
            "%{x}<br><b>Bearish regime begins</b>"
            "<br>Close: ₹%{customdata[0]:,.2f}"
            "<br>EMA 21: ₹%{customdata[1]:,.2f}"
            f"<br>EMA {TREND_PERIOD} low: ₹%{{customdata[2]:,.2f}}"
            "<br>Momentum: %{customdata[3]}<extra></extra>"
        ),
        legendrank=41,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x_values,
        y=plot_df["volume"],
        name="Volume",
        marker_color=volume_colors,
        hovertemplate="Volume: %{y:,.0f}<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=plot_df[f"adx_{ADX_PERIOD}"],
        mode="lines",
        name=f"ADX {ADX_PERIOD}",
        line=dict(color="#6d28d9", width=2.5),
        hovertemplate=f"ADX {ADX_PERIOD}: %{{y:.2f}}<extra></extra>",
        legendrank=50,
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=plot_df[f"atr_{ATR_PERIOD}"],
        mode="lines",
        name=f"ATR {ATR_PERIOD}",
        line=dict(color="#0284c7", width=2.2),
        fill="tozeroy",
        fillcolor="rgba(2, 132, 199, 0.10)",
        hovertemplate=f"ATR {ATR_PERIOD}: ₹%{{y:,.2f}}<extra></extra>",
        legendrank=51,
    ),
    row=4,
    col=1,
)
fig.add_hrect(
    y0=ADX_THRESHOLD,
    y1=100,
    fillcolor="rgba(109, 40, 217, 0.06)",
    line_width=0,
    row=3,
    col=1,
)
fig.add_hline(
    y=ADX_THRESHOLD,
    line_width=1.4,
    line_dash="dot",
    line_color="#6d28d9",
    annotation_text=f"ADX threshold {ADX_THRESHOLD}",
    annotation_position="top left",
    annotation_font_color="#6d28d9",
    row=3,
    col=1,
)

fig.update_layout(
    title=dict(
        text=(
            f"<b>{symbol_display} · 1-hour EMA Ribbons</b><br>"
            f"<sup>Latest: ₹{latest['close']:,.2f} · "
            f"Trend {latest['trend']} · Momentum {latest['momentum']} · "
            f"ADX {ADX_PERIOD} {latest[f'adx_{ADX_PERIOD}']:.2f} · "
            f"ATR {ATR_PERIOD} ₹{latest[f'atr_{ATR_PERIOD}']:,.2f}</sup>"
        ),
        x=0.01,
        xanchor="left",
    ),
    height=1050,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=75, r=35, t=125, b=70),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        font=dict(size=11),
    ),
    bargap=0.08,
    uirevision="netweb-ema-ribbon",
)
fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=x_values.tolist(),
    nticks=12,
    rangeslider_visible=False,
    showgrid=False,
)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(showticklabels=False, row=3, col=1)
fig.update_xaxes(title_text="Date / time (IST)", tickangle=-25, row=4, col=1)
fig.update_yaxes(title_text="Price (₹)", tickprefix="₹", tickformat=",.2f", row=1, col=1)
fig.update_yaxes(title_text="Volume", tickformat=".2s", rangemode="tozero", row=2, col=1)
fig.update_yaxes(
    title_text=f"ADX {ADX_PERIOD}",
    range=[0, 100],
    tickvals=sorted({0, 20, ADX_THRESHOLD, 40, 60, 80, 100}),
    row=3,
    col=1,
)
fig.update_yaxes(
    title_text=f"ATR {ATR_PERIOD} (₹)",
    tickprefix="₹",
    tickformat=",.2f",
    rangemode="tozero",
    row=4,
    col=1,
)
fig.show()

## Backtest setup

The test starts only after every ribbon/ADX/ATR/chandelier input is available. A signal observed at an hourly candle's close is filled at the **next candle's open**, which avoids using information before it exists. Costs are charged on both entry and exit, and a small spread/slippage is now applied to every fill.

The capital, costs, risk, and ATR bracket assumptions below are deliberately explicit and can be changed in one place. `EXIT_MODE` switches between the default fixed take-profit and a chandelier trailing stop meant to let winners run; `MAX_BARS_IN_TRADE` force-closes trades that go nowhere instead of tying up capital indefinitely.

In [6]:
INITIAL_CASH = 100_000
COMMISSION_RATE = 0.0002  # 0.02% per order; charged at entry and exit
SPREAD_RATE = 0.0005      # ~5 bps relative bid-ask spread/slippage per fill; set 0.0 to disable
MARGIN_REQUIREMENT = 1.0  # 1.0 means no leverage
RISK_PER_TRADE = 0.01     # Planned loss at the initial stop: 1% of equity
MAX_POSITION_FRACTION = 0.95
ATR_STOP_MULTIPLE = 2.0
REWARD_TO_RISK = 2.0      # Target distance = stop distance × this value; only used when EXIT_MODE == "fixed_tp"
ALLOW_SHORTS = True
EXIT_MODE = "fixed_tp"    # "fixed_tp" exits at a fixed reward:risk target (empirically better on
                           # this symbol below — see the exit-mode comparison after the backtest run).
                           # "trailing" lets a chandelier trail (see CHANDELIER_* above) ride the
                           # position instead; it's implemented and tunable but currently underperforms
                           # here with more whipsaw trades, so it's opt-in rather than the default.
MAX_BARS_IN_TRADE = 60     # Force-close a trade that hasn't hit its stop/trail/target after this
                           # many bars (~2 trading weeks on hourly bars); set to None to disable

indicator_columns = [
    *(f"ema_{period}_close" for period in MOMENTUM_PERIODS),
    f"ema_{TREND_PERIOD}_low",
    f"ema_{TREND_PERIOD}_close",
    f"ema_{TREND_PERIOD}_high",
    f"adx_{ADX_PERIOD}",
    f"atr_{ATR_PERIOD}",
    "chandelier_long",
    "chandelier_short",
]


def build_backtest_frame(price_df: pd.DataFrame, atr_period: int = ATR_PERIOD) -> pd.DataFrame:
    """Convert an indicator-enriched price frame into a backtesting.py-ready OHLCV frame."""
    frame = (
        price_df.dropna(subset=indicator_columns)
        .loc[
            :,
            [
                "open",
                "high",
                "low",
                "close",
                "volume",
                f"atr_{atr_period}",
                "chandelier_long",
                "chandelier_short",
                "bullish_crossover",
                "bearish_crossover",
            ],
        ]
        .rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
                f"atr_{atr_period}": "ATR",
                "chandelier_long": "ChandelierLong",
                "chandelier_short": "ChandelierShort",
                "bullish_crossover": "LongSignal",
                "bearish_crossover": "ShortSignal",
            }
        )
        .copy()
    )
    frame[["LongSignal", "ShortSignal"]] = frame[["LongSignal", "ShortSignal"]].astype(bool)
    frame.index.name = "timestamp"

    ohlcv_columns = ["Open", "High", "Low", "Close", "Volume"]
    if frame.empty:
        raise ValueError("No indicator-ready candles are available for the backtest.")
    if not frame.index.is_monotonic_increasing or not frame.index.is_unique:
        raise ValueError("Backtest timestamps must be sorted and unique.")
    if not np.isfinite(frame[ohlcv_columns + ["ATR"]]).all().all():
        raise ValueError("Backtest OHLCV/ATR values must all be finite.")
    if (frame[["Open", "High", "Low", "Close"]] <= 0).any().any():
        raise ValueError("Backtest prices must be positive.")
    if (frame["High"] < frame["Low"]).any():
        raise ValueError("Each candle's high must be at least its low.")

    return frame


backtest_data = build_backtest_frame(price)

print(
    f"Backtest input: {len(backtest_data):,} candles from "
    f"{backtest_data.index.min()} to {backtest_data.index.max()}"
)
print(
    f"Signals after warm-up: {backtest_data['LongSignal'].sum()} long and "
    f"{backtest_data['ShortSignal'].sum()} short"
)
display(backtest_data.head())

Backtest input: 2,105 candles from 2025-06-03 12:15:00+05:30 to 2026-08-19 15:15:00+05:30
Signals after warm-up: 88 long and 11 short


,Open,High,Low,Close,Volume,ATR,ChandelierLong,ChandelierShort,LongSignal,ShortSignal
timestamp,,,,,,,,,,
2025-06-03 12:15:00+05:30,315.75,316.25,314.10,314.40,47055,3.02,311.03,310.07,False,False
2025-06-03 13:15:00+05:30,314.60,315.70,314.40,314.70,27003,2.85,311.55,309.55,False,False
2025-06-03 14:15:00+05:30,314.85,315.50,314.05,314.25,34412,2.71,311.97,309.13,False,False
2025-06-03 15:15:00+05:30,314.15,316.00,314.05,314.70,38714,2.63,312.20,308.90,False,False
2025-06-04 09:15:00+05:30,315.00,316.15,311.55,313.90,154296,2.83,311.61,309.49,False,False


## Strategy and execution rules

- Enter long on a bullish ribbon event and short on a bearish ribbon event.
- Ignore repeated same-direction events while already holding that direction; an opposite event reverses the position.
- Size whole shares so the initial 2-ATR stop risks at most 1% of current equity, capped at 95% gross allocation.
- **`EXIT_MODE = "fixed_tp"` (default):** attaches a take-profit at twice the initial risk (4 ATR with the defaults), as in the original version of this notebook. **`EXIT_MODE = "trailing"`:** no fixed target — a chandelier stop (highest high in the last `CHANDELIER_LOOKBACK` bars, minus `CHANDELIER_ATR_MULTIPLE × ATR`) tightens every bar and only ever moves in the trade's favor, letting winners run until the trail is hit or an opposite regime event reverses the position. The exit-mode comparison below shows `trailing` currently trades more often and does worse here — it's kept as a tunable, testable option rather than the default.
- A trade open for `MAX_BARS_IN_TRADE` bars without hitting its stop, trail, or target is force-closed, so capital isn't tied up indefinitely in a trade that's going nowhere.
- Market orders fill on the next observed candle's open, with a small spread applied to every fill. A stop can fill worse than its trigger when price gaps.
- Keep only one open position and close any final open trade on the last candle so it appears in the statistics.

In [7]:
class EmaRibbonStrategy(Strategy):
    risk_per_trade = RISK_PER_TRADE
    max_position_fraction = MAX_POSITION_FRACTION
    atr_stop_multiple = ATR_STOP_MULTIPLE
    reward_to_risk = REWARD_TO_RISK
    allow_shorts = ALLOW_SHORTS
    exit_mode = EXIT_MODE
    max_bars_in_trade = MAX_BARS_IN_TRADE

    def init(self):
        if not 0 < self.risk_per_trade <= 1:
            raise ValueError("risk_per_trade must be in (0, 1].")
        if not 0 < self.max_position_fraction <= 1:
            raise ValueError("max_position_fraction must be in (0, 1].")
        if self.atr_stop_multiple <= 0:
            raise ValueError("atr_stop_multiple must be positive.")
        if self.reward_to_risk <= 0:
            raise ValueError("reward_to_risk must be positive.")
        if self.exit_mode not in ("fixed_tp", "trailing"):
            raise ValueError("exit_mode must be 'fixed_tp' or 'trailing'.")
        if self.max_bars_in_trade is not None and self.max_bars_in_trade <= 0:
            raise ValueError("max_bars_in_trade must be positive when set.")

    def _position_size(self, reference_price: float, stop_distance: float) -> int:
        risk_budget = self.equity * self.risk_per_trade
        risk_limited_units = int(risk_budget // stop_distance)

        # Keep a cash buffer so a next-open gap and transaction costs do not
        # cause an otherwise valid order to be rejected for insufficient margin.
        cost_buffer = 1 + COMMISSION_RATE + SPREAD_RATE
        allocation = self.equity * self.max_position_fraction
        allocation_limited_units = int(allocation // (reference_price * cost_buffer))
        return max(0, min(risk_limited_units, allocation_limited_units))

    def _submit_entry(self, *, is_long: bool, reference_price: float, atr: float) -> None:
        stop_distance = atr * self.atr_stop_multiple
        units = self._position_size(reference_price, stop_distance)
        if units < 1:
            return

        take_profit = None
        if is_long:
            stop_loss = reference_price - stop_distance
            if stop_loss <= 0:
                return
            if self.exit_mode == "fixed_tp":
                take_profit = reference_price + stop_distance * self.reward_to_risk
            self.buy(size=units, sl=stop_loss, tp=take_profit, tag="Bullish ribbon")
        else:
            stop_loss = reference_price + stop_distance  # Adjusted stop loss for shorts
            if self.exit_mode == "fixed_tp":
                take_profit = reference_price - stop_distance * self.reward_to_risk
                if take_profit <= 0:
                    return
            self.sell(size=units, sl=stop_loss, tp=take_profit, tag="Bearish ribbon")

    def _trail_stops(self) -> None:
        """Tighten (never loosen) each open trade's stop toward the chandelier trail."""
        if self.exit_mode != "trailing":
            return
        chandelier_long = float(self.data.ChandelierLong[-1])
        chandelier_short = float(self.data.ChandelierShort[-1])
        for trade in self.trades:
            if trade.is_long and np.isfinite(chandelier_long):
                trade.sl = max(trade.sl, chandelier_long)
            elif trade.is_short and np.isfinite(chandelier_short):
                trade.sl = min(trade.sl, chandelier_short)

    def _close_stale_trades(self) -> None:
        if self.max_bars_in_trade is None:
            return
        current_bar = len(self.data) - 1
        for trade in self.trades:
            if current_bar - trade.entry_bar >= self.max_bars_in_trade:
                trade.close()

    def next(self):
        self._trail_stops()
        self._close_stale_trades()

        long_signal = bool(self.data.LongSignal[-1])
        short_signal = bool(self.data.ShortSignal[-1]) and self.allow_shorts
        if long_signal == short_signal:  # Neither signal, or an invalid simultaneous signal.
            return

        reference_price = float(self.data.Close[-1])
        atr = float(self.data.ATR[-1])
        if not np.isfinite(reference_price) or not np.isfinite(atr) or atr <= 0:
            return

        if long_signal and not self.position.is_long:
            self._submit_entry(is_long=True, reference_price=reference_price, atr=atr)
        elif short_signal and not self.position.is_short:
            self._submit_entry(is_long=False, reference_price=reference_price, atr=atr)

## Run the backtest

`exclusive_orders=True` makes a new opposite order close/reverse the existing trade. `trade_on_close=False` preserves next-open execution, and `finalize_trades=True` includes the final open position in the report.

In [8]:
def run_symbol_backtest(
    backtest_frame: pd.DataFrame,
    *,
    cash: float = INITIAL_CASH,
    spread: float = SPREAD_RATE,
    commission: float = COMMISSION_RATE,
    margin: float = MARGIN_REQUIREMENT,
    risk_per_trade: float = RISK_PER_TRADE,
    max_position_fraction: float = MAX_POSITION_FRACTION,
    atr_stop_multiple: float = ATR_STOP_MULTIPLE,
    reward_to_risk: float = REWARD_TO_RISK,
    allow_shorts: bool = ALLOW_SHORTS,
    exit_mode: str = EXIT_MODE,
    max_bars_in_trade: int | None = MAX_BARS_IN_TRADE,
):
    """Run EmaRibbonStrategy over a pre-built backtest frame and return (Backtest, stats)."""
    bt = Backtest(
        backtest_frame,
        EmaRibbonStrategy,
        cash=cash,
        spread=spread,
        commission=commission,
        margin=margin,
        trade_on_close=False,
        hedging=False,
        exclusive_orders=True,
        finalize_trades=True,
    )
    run_stats = bt.run(
        risk_per_trade=risk_per_trade,
        max_position_fraction=max_position_fraction,
        atr_stop_multiple=atr_stop_multiple,
        reward_to_risk=reward_to_risk,
        allow_shorts=allow_shorts,
        exit_mode=exit_mode,
        max_bars_in_trade=max_bars_in_trade,
    )
    return bt, run_stats


backtest, stats = run_symbol_backtest(backtest_data)

summary_keys = [
    "Start",
    "End",
    "Duration",
    "Exposure Time [%]",
    "Equity Final [$]",
    "Equity Peak [$]",
    "Commissions [$]",
    "Return [%]",
    "Buy & Hold Return [%]",
    "Return (Ann.) [%]",
    "Volatility (Ann.) [%]",
    "CAGR [%]",
    "Sharpe Ratio",
    "Sortino Ratio",
    "Calmar Ratio",
    "Max. Drawdown [%]",
    "Max. Drawdown Duration",
    "# Trades",
    "Win Rate [%]",
    "Best Trade [%]",
    "Worst Trade [%]",
    "Avg. Trade [%]",
    "Profit Factor",
    "Expectancy [%]",
    "SQN",
]
summary = stats.loc[[key for key in summary_keys if key in stats.index]].copy()
summary = summary.rename(
    index={
        "Equity Final [$]": "Equity Final [₹]",
        "Equity Peak [$]": "Equity Peak [₹]",
        "Commissions [$]": "Commissions [₹]",
    }
)
display(summary.to_frame(name="Value"))

,Value
Start,2025-06-03 12:15:00+05:30
End,2026-08-19 15:15:00+05:30
Duration,442 days 03:00:00
Exposure Time [%],42.14
Equity Final [₹],"118,668.42"
Equity Peak [₹],"120,706.99"
Commissions [₹],688.76
Return [%],18.67
Buy & Hold Return [%],358.02
Return (Ann.) [%],15.41


## Exit mode comparison: fixed take-profit vs. trailing

The summary above uses `EXIT_MODE = "fixed_tp"`. This reruns the identical signals and risk sizing with the chandelier trailing stop instead, so the two exit rules can be compared directly on the same symbol and history. On this symbol the trailing stop currently trades more often (more whipsaws) and comes out behind — worth tuning `CHANDELIER_LOOKBACK`/`CHANDELIER_ATR_MULTIPLE` and re-checking via the walk-forward section below before relying on it.

In [9]:
_, trailing_stats = run_symbol_backtest(backtest_data, exit_mode="trailing")

present_summary_keys = [key for key in summary_keys if key in stats.index]
exit_mode_comparison = pd.DataFrame(
    {
        "Fixed take-profit (current)": stats.loc[present_summary_keys],
        "Trailing": trailing_stats.loc[present_summary_keys],
    }
)
display(exit_mode_comparison)

,Fixed take-profit (current),Trailing
Start,2025-06-03 12:15:00+05:30,2025-06-03 12:15:00+05:30
End,2026-08-19 15:15:00+05:30,2026-08-19 15:15:00+05:30
Duration,442 days 03:00:00,442 days 03:00:00
Exposure Time [%],42.14,33.97
Equity Final [$],"118,668.42","105,445.18"
Equity Peak [$],"120,706.99","106,762.88"
Commissions [$],688.76,804.16
Return [%],18.67,5.45
Buy & Hold Return [%],358.02,358.02
Return (Ann.) [%],15.41,4.54


## Trade log and direction breakdown

PnL and return values below are net of the configured entry and exit commissions. Exit reasons are inferred from the recorded bracket levels and final bar; remaining exits are signal reversals.

In [10]:
trades = stats["_trades"].copy()

if trades.empty:
    print("No trades were completed with the current settings.")
    trade_log = trades.copy()
    direction_summary = pd.DataFrame()
else:
    is_long = trades["Size"].gt(0)
    price_tolerance = np.maximum(trades["ExitPrice"].abs() * 1e-9, 1e-9)
    stop_exit = (
        (is_long & trades["ExitPrice"].le(trades["SL"] + price_tolerance))
        | (~is_long & trades["ExitPrice"].ge(trades["SL"] - price_tolerance))
    )
    target_exit = (
        (is_long & trades["ExitPrice"].ge(trades["TP"] - price_tolerance))
        | (~is_long & trades["ExitPrice"].le(trades["TP"] + price_tolerance))
    )
    final_exit = trades["ExitBar"].eq(len(backtest_data) - 1)

    trades["Direction"] = np.where(is_long, "Long", "Short")
    trades["ExitReason"] = np.select(
        [stop_exit, target_exit, final_exit],
        ["Stop loss", "Take profit", "End of data"],
        default="Signal reversal",
    )
    trades["Return [%]"] = trades["ReturnPct"] * 100

    trade_log = trades[
        [
            "Direction",
            "EntryTime",
            "ExitTime",
            "Duration",
            "Size",
            "EntryPrice",
            "ExitPrice",
            "SL",
            "TP",
            "PnL",
            "Commission",
            "Return [%]",
            "ExitReason",
        ]
    ].rename(
        columns={
            "EntryPrice": "Entry [₹]",
            "ExitPrice": "Exit [₹]",
            "SL": "Stop [₹]",
            "TP": "Target [₹]",
            "PnL": "PnL [₹]",
            "Commission": "Commission [₹]",
        }
    )

    direction_summary = (
        trades.groupby("Direction", sort=False)
        .agg(
            Trades=("PnL", "size"),
            Net_PnL_INR=("PnL", "sum"),
            Win_Rate_pct=("PnL", lambda values: values.gt(0).mean() * 100),
            Average_Return_pct=("ReturnPct", lambda values: values.mean() * 100),
            Average_Duration=("Duration", "mean"),
        )
        .rename(
            columns={
                "Net_PnL_INR": "Net PnL [₹]",
                "Win_Rate_pct": "Win Rate [%]",
                "Average_Return_pct": "Average Return [%]",
                "Average_Duration": "Average Duration",
            }
        )
    )

    print(f"Completed trades: {len(trade_log)}")
    display(direction_summary)
    display(trade_log)

Completed trades: 50


,Trades,Net PnL [₹],Win Rate [%],Average Return [%],Average Duration
Direction,,,,,
Short,8,"-5,938.19",12.50,-1.89,1 days 20:52:30
Long,42,"24,606.60",50.00,1.97,3 days 23:43:34.285714


,Direction,EntryTime,ExitTime,Duration,Size,Entry [₹],Exit [₹],Stop [₹],Target [₹],PnL [₹],Commission [₹],Return [%],ExitReason
0,Short,2025-06-04 14:15:00+05:30,2025-06-09 09:15:00+05:30,4 days 19:00:00,-180,309.10,314.80,314.80,298.15,"-1,049.49",22.46,-1.89,Stop loss
1,Long,2025-06-25 11:15:00+05:30,2025-07-08 09:15:00+05:30,12 days 22:00:00,113,331.62,329.80,322.75,348.85,-220.12,14.95,-0.59,Signal reversal
2,Long,2025-07-10 11:15:00+05:30,2025-07-14 09:15:00+05:30,3 days 22:00:00,127,332.17,347.31,324.12,347.31,"1,906.31",17.26,4.52,Take profit
3,Long,2025-07-14 11:15:00+05:30,2025-07-16 11:15:00+05:30,2 days 00:00:00,94,350.02,339.53,339.53,371.39,-999.44,12.96,-3.04,Stop loss
4,Long,2025-07-17 11:15:00+05:30,2025-07-18 09:15:00+05:30,0 days 22:00:00,108,353.43,344.03,344.03,371.70,"-1,030.33",15.06,-2.70,Stop loss
5,Long,2025-07-29 14:15:00+05:30,2025-07-30 10:15:00+05:30,0 days 20:00:00,135,344.87,337.35,337.35,359.24,"-1,033.43",18.42,-2.22,Stop loss
6,Long,2025-07-30 15:15:00+05:30,2025-08-04 11:15:00+05:30,4 days 20:00:00,99,346.72,366.30,336.75,366.30,"1,923.91",14.12,5.60,Take profit
7,Long,2025-08-05 15:15:00+05:30,2025-08-08 11:15:00+05:30,2 days 20:00:00,44,388.69,433.11,366.20,433.11,"1,947.01",7.23,11.38,Take profit
8,Long,2025-08-14 10:15:00+05:30,2025-08-19 15:15:00+05:30,5 days 05:00:00,62,417.31,449.54,401.11,449.54,"1,987.29",10.75,7.68,Take profit
9,Long,2025-08-22 13:15:00+05:30,2025-08-22 14:15:00+05:30,0 days 01:00:00,81,429.31,416.63,416.63,454.79,"-1,041.00",13.70,-2.99,Stop loss


## Equity, benchmark, and drawdown

The buy-and-hold line invests the same initial cash at the first indicator-ready close. It is a price-return benchmark and does not include dividends or its own transaction costs.

In [11]:
symbol_display = SYMBOLS[PRIMARY_SYMBOL]["display"]

equity_curve = stats["_equity_curve"].copy()
benchmark_close = backtest_data["Close"].reindex(equity_curve.index)
equity_curve["BuyAndHoldEquity"] = (
    INITIAL_CASH * benchmark_close / benchmark_close.iloc[0]
)
equity_curve["Drawdown [%]"] = -100 * equity_curve["DrawdownPct"]

performance_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.72, 0.28],
    vertical_spacing=0.06,
    subplot_titles=("Strategy equity vs buy and hold", "Strategy drawdown"),
)
performance_fig.add_trace(
    go.Scatter(
        x=equity_curve.index,
        y=equity_curve["Equity"],
        mode="lines",
        name="EMA ribbon strategy",
        line=dict(color="#2563eb", width=2.4),
        hovertemplate="Equity: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
performance_fig.add_trace(
    go.Scatter(
        x=equity_curve.index,
        y=equity_curve["BuyAndHoldEquity"],
        mode="lines",
        name="Buy and hold",
        line=dict(color="#64748b", width=1.8, dash="dash"),
        hovertemplate="Buy and hold: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
performance_fig.add_trace(
    go.Scatter(
        x=equity_curve.index,
        y=equity_curve["Drawdown [%]"],
        mode="lines",
        name="Drawdown",
        line=dict(color="#dc2626", width=1.8),
        fill="tozeroy",
        fillcolor="rgba(220, 38, 38, 0.14)",
        hovertemplate="Drawdown: %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)
performance_fig.update_layout(
    title=dict(text=f"<b>{symbol_display} EMA Ribbon Backtest</b>", x=0.01),
    height=760,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=75, r=30, t=90, b=55),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
performance_fig.update_yaxes(title_text="Equity (₹)", tickprefix="₹", row=1, col=1)
performance_fig.update_yaxes(title_text="Drawdown (%)", ticksuffix="%", row=2, col=1)
performance_fig.update_xaxes(title_text="Date / time (IST)", row=2, col=1)
performance_fig.show()

## Backtesting.py trade chart

This package-native chart overlays fills and bracket exits on OHLC candles. Set the switch to `False` when running unattended if only the tabular results and Plotly performance chart are needed.

In [12]:
SHOW_INTERACTIVE_TRADE_CHART = True

if SHOW_INTERACTIVE_TRADE_CHART:
    backtest.plot(
        results=stats,
        plot_equity=True,
        plot_return=False,
        plot_pl=True,
        plot_volume=True,
        plot_drawdown=True,
        plot_trades=True,
        smooth_equity=False,
        relative_equity=False,
        superimpose=False,
        resample=False,
        reverse_indicators=False,
        show_legend=True,
        open_browser=False,
    )

/home/amit/anaconda3/envs/visualizer/lib/python3.11/site-packages/bokeh/util/serialization.py:247: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


## Multi-symbol basket backtest

The single-symbol walkthrough above has only ~50 completed trades — too few to trust its Sharpe/Calmar/SQN in isolation. Re-running the identical, unmodified `EmaRibbonStrategy` across every symbol in `data/raw/` gives a larger combined trade sample and shows whether an equal-weight basket reduces drawdown through diversification versus trading one stock alone.

Each symbol gets its own slice of capital (`INITIAL_CASH / len(SYMBOLS)`) and its own independent position; the combined equity curve sums the symbols' equity paths reindexed onto one shared hourly timeline. Note `atherenerg`'s indicator-ready history starts later than the other four symbols (see the per-symbol date ranges above) — before a symbol's own data begins, its slice of capital is treated as idle cash at its starting allocation, not invested.

In [13]:
BASKET_INITIAL_CASH = INITIAL_CASH / len(SYMBOLS)

basket_results = {}
for symbol_key in SYMBOLS:
    symbol_price = add_ema_ribbon_indicators(load_symbol_data(symbol_key))
    symbol_frame = build_backtest_frame(symbol_price)
    symbol_backtest, symbol_stats = run_symbol_backtest(symbol_frame, cash=BASKET_INITIAL_CASH)
    basket_results[symbol_key] = {"backtest_frame": symbol_frame, "stats": symbol_stats}

basket_summary = pd.DataFrame(
    {
        SYMBOLS[symbol_key]["display"]: {
            "Return [%]": result["stats"]["Return [%]"],
            "Buy & Hold Return [%]": result["stats"]["Buy & Hold Return [%]"],
            "Max. Drawdown [%]": result["stats"]["Max. Drawdown [%]"],
            "Max. Drawdown Duration": result["stats"]["Max. Drawdown Duration"],
            "Sharpe Ratio": result["stats"]["Sharpe Ratio"],
            "Calmar Ratio": result["stats"]["Calmar Ratio"],
            "# Trades": result["stats"]["# Trades"],
            "Win Rate [%]": result["stats"]["Win Rate [%]"],
            "Profit Factor": result["stats"]["Profit Factor"],
        }
        for symbol_key, result in basket_results.items()
    }
).T
display(basket_summary)

,Return [%],Buy & Hold Return [%],Max. Drawdown [%],Max. Drawdown Duration,Sharpe Ratio,Calmar Ratio,# Trades,Win Rate [%],Profit Factor
Adani Green Energy,9.67,-19.64,-8.57,268 days 00:00:00,0.35,0.42,108,37.04,1.07
Ather Energy,17.60,358.02,-5.67,287 days 01:00:00,1.33,2.56,50,44.00,1.78
HDFC Bank,-11.37,-15.35,-19.67,769 days 05:00:00,-0.49,-0.23,110,34.55,0.98
Netweb Technologies,25.00,349.20,-8.96,292 days 03:00:00,0.97,1.00,118,41.53,1.41
Reliance Industries,-11.20,1.39,-18.60,439 days 23:00:00,-0.49,-0.24,107,32.71,0.79


In [14]:
combined_index = None
for result in basket_results.values():
    idx = result["backtest_frame"].index
    combined_index = idx if combined_index is None else combined_index.union(idx)

combined_equity = pd.Series(0.0, index=combined_index)
combined_buy_hold = pd.Series(0.0, index=combined_index)
for symbol_key, result in basket_results.items():
    equity = (
        result["stats"]["_equity_curve"]["Equity"]
        .reindex(combined_index)
        .ffill()
        .fillna(BASKET_INITIAL_CASH)
    )
    combined_equity += equity

    close = result["backtest_frame"]["Close"]
    buy_hold = (
        (BASKET_INITIAL_CASH * close / close.iloc[0])
        .reindex(combined_index)
        .ffill()
        .fillna(BASKET_INITIAL_CASH)
    )
    combined_buy_hold += buy_hold

combined_drawdown = 100 * (combined_equity / combined_equity.cummax() - 1)

basket_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.72, 0.28],
    vertical_spacing=0.06,
    subplot_titles=("Basket equity vs combined buy and hold", "Basket drawdown"),
)
basket_fig.add_trace(
    go.Scatter(
        x=combined_equity.index,
        y=combined_equity,
        mode="lines",
        name="EMA ribbon basket",
        line=dict(color="#2563eb", width=2.4),
        hovertemplate="Equity: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
basket_fig.add_trace(
    go.Scatter(
        x=combined_buy_hold.index,
        y=combined_buy_hold,
        mode="lines",
        name="Buy and hold basket",
        line=dict(color="#64748b", width=1.8, dash="dash"),
        hovertemplate="Buy and hold: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
basket_fig.add_trace(
    go.Scatter(
        x=combined_drawdown.index,
        y=combined_drawdown,
        mode="lines",
        name="Drawdown",
        line=dict(color="#dc2626", width=1.8),
        fill="tozeroy",
        fillcolor="rgba(220, 38, 38, 0.14)",
        hovertemplate="Drawdown: %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)
basket_fig.update_layout(
    title=dict(text="<b>EMA Ribbon Basket Backtest (5 symbols, equal-weight)</b>", x=0.01),
    height=760,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=75, r=30, t=90, b=55),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
basket_fig.update_yaxes(title_text="Equity (₹)", tickprefix="₹", row=1, col=1)
basket_fig.update_yaxes(title_text="Drawdown (%)", ticksuffix="%", row=2, col=1)
basket_fig.update_xaxes(title_text="Date / time (IST)", row=2, col=1)
basket_fig.show()

basket_max_drawdown = combined_drawdown.min()
print(
    f"Basket max drawdown: {basket_max_drawdown:.2f}% "
    f"(vs single-symbol {stats['Max. Drawdown [%]']:.2f}% for {SYMBOLS[PRIMARY_SYMBOL]['display']})"
)

Basket max drawdown: -5.15% (vs single-symbol -5.82% for Ather Energy)


## Walk-forward validation

The original in-sample grid search below scores parameters on the exact same history it then reports them against, which risks overfitting `atr_stop_multiple`, `reward_to_risk`, `exit_mode`, and the regime filter to this one dataset. This section instead walks forward: for each fold, parameters (including a choice between `fixed_tp` and `trailing`) are selected only on a training window, then evaluated once, unmodified, on the following out-of-sample test window.

Each fold's test result is a standalone backtest starting from `INITIAL_CASH` (not chained into one continuous equity curve), so folds are directly comparable to each other and to the full-history run above, but should not be summed into a single out-of-sample return. Disabled by default (`RUN_WALK_FORWARD = False`) because each fold refits a small parameter grid, which takes noticeably longer than a single backtest.

In [15]:
RUN_WALK_FORWARD = False
WALK_FORWARD_SYMBOL = PRIMARY_SYMBOL
WALK_FORWARD_FOLDS = 3
WALK_FORWARD_TEST_FRACTION = 0.25   # Fraction of each fold's window held out for testing
WALK_FORWARD_MIN_TRAIN_TRADES = 10  # Relaxed vs. the 20-trade floor a full-history search would use


def walk_forward_score(candidate_stats: pd.Series) -> float:
    if candidate_stats["# Trades"] < WALK_FORWARD_MIN_TRAIN_TRADES or not np.isfinite(candidate_stats["SQN"]):
        return -np.inf
    return float(candidate_stats["SQN"])


def walk_forward_validate(symbol_key: str, n_folds: int, test_fraction: float) -> pd.DataFrame:
    """Expanding-window walk-forward: fit a small parameter grid on each train window, then
    report one out-of-sample run on the following test window."""
    full_price = add_ema_ribbon_indicators(load_symbol_data(symbol_key))
    full_frame = build_backtest_frame(full_price)

    total_bars = len(full_frame)
    test_size = max(int(total_bars * test_fraction / n_folds), 1)
    fold_records = []

    for fold in range(1, n_folds + 1):
        test_end = int(total_bars * fold / n_folds)
        test_start = max(test_end - test_size, 0)
        train_frame = full_frame.iloc[:test_start]
        test_frame = full_frame.iloc[test_start:test_end]
        if train_frame.empty or test_frame.empty:
            continue

        train_backtest = Backtest(
            train_frame,
            EmaRibbonStrategy,
            cash=INITIAL_CASH,
            spread=SPREAD_RATE,
            commission=COMMISSION_RATE,
            margin=MARGIN_REQUIREMENT,
            trade_on_close=False,
            hedging=False,
            exclusive_orders=True,
            finalize_trades=True,
        )
        train_stats = train_backtest.optimize(
            atr_stop_multiple=[1.5, 2.0, 2.5],
            reward_to_risk=[1.5, 2.0, 2.5],
            exit_mode=["fixed_tp", "trailing"],
            maximize=walk_forward_score,
        )
        if train_stats["# Trades"] < WALK_FORWARD_MIN_TRAIN_TRADES or not np.isfinite(train_stats["SQN"]):
            continue  # No parameter combination cleared the minimum-trade bar on this fold.

        best_strategy = train_stats["_strategy"]
        chosen_params = {
            "atr_stop_multiple": best_strategy.atr_stop_multiple,
            "reward_to_risk": best_strategy.reward_to_risk,
            "exit_mode": best_strategy.exit_mode,
        }

        _, test_stats = run_symbol_backtest(test_frame, **chosen_params)

        fold_records.append(
            {
                "Fold": fold,
                "Train start": train_frame.index.min(),
                "Train end": train_frame.index.max(),
                "Test start": test_frame.index.min(),
                "Test end": test_frame.index.max(),
                "Chosen atr_stop_multiple": chosen_params["atr_stop_multiple"],
                "Chosen reward_to_risk": chosen_params["reward_to_risk"],
                "Chosen exit_mode": chosen_params["exit_mode"],
                "Test Return [%]": test_stats["Return [%]"],
                "Test Max. Drawdown [%]": test_stats["Max. Drawdown [%]"],
                "Test # Trades": test_stats["# Trades"],
                "Test Sharpe Ratio": test_stats["Sharpe Ratio"],
            }
        )

    return pd.DataFrame(fold_records)


if RUN_WALK_FORWARD:
    walk_forward_results = walk_forward_validate(
        WALK_FORWARD_SYMBOL, WALK_FORWARD_FOLDS, WALK_FORWARD_TEST_FRACTION
    )
    if walk_forward_results.empty:
        print("No fold produced a valid train/test split with enough trades.")
    else:
        display(walk_forward_results)
        print(
            f"Out-of-sample mean return: {walk_forward_results['Test Return [%]'].mean():.2f}% | "
            f"worst fold drawdown: {walk_forward_results['Test Max. Drawdown [%]'].min():.2f}%"
        )